# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will learn how to load structured data defined by a Croissant schema, review available record sets and fields, extract data, and perform exploratory analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}\nIdentifier: {getattr(metadata, 'identifier', None)}\n")
if hasattr(metadata, 'author'):
    print(f"Authors: {[getattr(a, '@id', str(a)) for a in metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In Croissant, structured datasets are organized into record sets. Let's enumerate all record sets, showing their `@id`, name, and the associated fields/columns (`@id`s only).

In [ ]:
print("Available record sets and their fields:\n")
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in metadata. Trying fallback for possible versions...")
    # Sometimes older/alternate versions may use `.record_set` or `.recordSet`.
    record_sets = getattr(metadata, 'record_set', None) or getattr(metadata, 'recordSet', None)
    if not record_sets:
        print("No record sets found.")
        record_sets = []
if isinstance(record_sets, dict):
    # Croissant guarantees a list, but if only a single, it's a dict
    record_sets = [record_sets]
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"RecordSet: {rs_id}  Name: {rs_name}")
    rs_fields = getattr(rs, 'fields', None)
    if rs_fields is not None and isinstance(rs_fields, (list, tuple)):
        print("  Fields:")
        for field in rs_fields:
            field_id = getattr(field, '@id', None) or getattr(field, 'id', field)
            print(f"    - {field_id}")
    rs_columns = getattr(rs, 'columns', None)
    if rs_columns:
        print("  Columns:")
        for col in rs_columns:
            col_id = getattr(col, '@id', None) or getattr(col, 'id', col)
            print(f"    - {col_id}")
    record_set_ids.append(rs_id)
if not record_set_ids:
    print("No record_sets were detected. Check the Croissant schema for structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we load all records from each record set (referencing them by their `@id`). If only one record set exists, we'll extract from that.


In [ ]:
if not record_set_ids:
    # Manual fallback for known dataset
    record_set_ids = ['https://sen.science/doi/10.71728/senscience.vq4a-28xa/rs/proteins']  # <-- Replace with correct from overview
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if not records:
            print(f"No records found for {record_set_id}")
            dataframes[record_set_id] = pd.DataFrame()
        else:
            dataframes[record_set_id] = pd.DataFrame(records)
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")
        dataframes[record_set_id] = pd.DataFrame()
# Preview the first table
primary_id = record_set_ids[0]
print(f"Available columns for {primary_id}:\n", dataframes[primary_id].columns.tolist())
dataframes[primary_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by categorical variables.

As an example, we'll select a numeric field, filter for high values, normalize it, and group by a potential categorical field if one exists.

In [ ]:
import numpy as np
df = dataframes[primary_id]
print(f"DataFrame shape: {df.shape}")
if isinstance(df, pd.DataFrame) and not df.empty:
    # Guess possible numeric fields (float/int dtypes or column names)
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or 'count' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower()]
    print(f"Candidate numeric fields: {numeric_candidates}")
    # Use the first numeric field found
    numeric_field = numeric_candidates[0] if numeric_candidates else df.select_dtypes('number').columns[0]
    print(f"Using numeric field: {numeric_field}")
    # Use median for threshold if distribution is wide
    threshold = df[numeric_field].median() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Number of records with {numeric_field} > {threshold}: {len(filtered_df)}")
    from warnings import filterwarnings
    filterwarnings('ignore', category=FutureWarning)
    filtered_df = filtered_df.copy()  # avoid SettingWithCopyWarning
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())
    
    # Try grouping by a likely field (e.g. 'Modification', or first object/string field)
    group_candidates = [col for col in df.columns if col != numeric_field and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col]))]
    group_field = group_candidates[0] if group_candidates else None
    if group_field is not None:
        print(f"Grouping by {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show histogram and boxplot for numeric field
if isinstance(df, pd.DataFrame) and not df.empty and numeric_field in df.columns:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
    
    # If a normalized column exists, plot scatter
    if norm_col in df.columns:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=df[numeric_field], y=df[norm_col])
        plt.title(f"{numeric_field} vs {norm_col}")
        plt.xlabel(numeric_field)
        plt.ylabel(norm_col)
        plt.show()
else:
    print("No visualization available: Data or numeric field missing.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load FAIR² dataset metadata and records via the Croissant schema URL. We explored the schema, loaded a primary record set by its `@id`, and performed a simple EDA, normalization, and visualization of a numeric field. For more advanced analysis, consult the dataset schema for semantic field types and relationships.
